# Benchmarking models for oxygen fugacity related variables

This notebook benchmarks the models for oxygen fugacity related variables in VolFe where possible.

## Python set-up

In [1]:
import VolFe as vf
import math
import pandas as pd
import Thermobar as tb

## Models for FMQ

option = FMQbuffer, function = FMQ

### 'ONeill87' O'Neill (1987) AmMin 72(1-2):67-75

Appendix A. Supplementary material - Supplementary data 2. Tab = Table S6 S redox calculator (sulfate,KSOg2=ONeill22.xlsx).

Cell R12: 0.13 [DFMQ]

Cell S12: -8.29 [logfO2]

Matches to 2 decimal places. Note spreadsheet uses +273 to convert to K, rather than 273.15 used in VolFe so T in spreadsheet = 1200.15 'C


In [2]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # DFMQ

my_models = [["FMQbuffer", "ONeill87"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'FMQ',my_models))

-8.294378372874451

### 'Frost91' Frost (1991) in "Oxide Minerals: Petrologic and Magnetic Significance"

Comparison with results from Thermobar (Wieser et al., 2022) - values match perfectly.

Define input conditions:

In [3]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # DFMQ

Run using VolFe

In [4]:
my_models = [["FMQbuffer", "Frost91"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'FMQ',my_models))

-8.096212368054847

Run using Thermobar

In [5]:
tb.calculate_logfo2_from_buffer_pos(T_K=PT["T"]+273.15,P_kbar=PT['P']/1000.,fo2_offset=D,buffer='QFM')

-8.096212368054847

## Models for NNO

option = NNObuffer, function = NNO

### 'Frost91' Frost (1991) in "Oxide Minerals: Petrologic and Magnetic Significance"

Comparison with results from Thermobar (Wieser et al., 2022) - values match perfectly.

Define input conditions:

In [6]:
PT = {"P":1000.}
PT["T"]=1200.
D = 0.13 # NNO

Run using VolFe

In [7]:
my_models = [["NNObuffer", "Frost91"]]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.Dbuffer2fO2(PT,D,'NNO',my_models))

-7.401725893493535

Run using Thermobar

In [8]:
tb.calculate_logfo2_from_buffer_pos(T_K=PT["T"]+273.15,P_kbar=PT['P']/1000.,fo2_offset=D,buffer='NNO')

-7.401725893493535

## Models for fO2-Fe3+/FeT

option = fO2, function = fO2

The following models do not have material available in the original papers for benchmarking:
- 'Kress91A' Eq. (A-5, A-6) in Kress and Carmichael (1991) CMP 108:82-92 doi:10.1007/BF00307328 (although the value is similar to 'Kress91' as expected)
- 'Borisov18' Eq. (4) from Borisov et al. (2018) CMP 173:98 doi:10.1007/s00410-018-1524-8   

### 'Kress91' Eq. (7) in Kress and Carmichael (1991) CMP 108:82-92

Comparison with results from Thermobar (Wieser et al., 2022) - values match to two signifcant figures.

In [12]:
my_analysis = {
        "Sample": "Hawaiian basalt",
        "SiO2": 51.29,  # wt%
        "TiO2": 2.50,  # wt%
        "Al2O3": 13.70,  # wt%
        "FeOT": 11.04,  # wt%
        "MnO": 0.02,  # wt%
        "MgO": 6.70,  # wt%
        "CaO": 11.03,  # wt%
        "Na2O": 2.27,  # wt%
        "K2O": 0.43,  # wt%
        "P2O5": 0.,  # wt%
        "H2O": 3.,  # wt%
        "CO2ppm": 1000.,  # ppm
        "STppm": 0.,  # ppm
        "Xppm": 0.0,  # ppm
        "Fe3FeT": 0.1,
    }

my_analysis = pd.DataFrame(my_analysis, index=[0])

PT = {"P":1000.}
PT["T"]=1200.

melt_wf=vf.melt_comp(0.,my_analysis)
melt_wf['CO2'] = my_analysis.loc[0.,"CO2ppm"]/1000000.
melt_wf["H2OT"] = my_analysis.loc[0,"H2O"]/100.
melt_wf["H2O"] = my_analysis.loc[0,"H2O"]/100.
melt_wf['ST'] = my_analysis.loc[0.,"STppm"]/1000000.
melt_wf['CT'] = (melt_wf['CO2']/vf.species.loc['CO2','M'])*vf.species.loc['C','M']
melt_wf['HT'] = (melt_wf['H2OT']/vf.species.loc['H2O','M'])*(2.*vf.species.loc['H','M'])
melt_wf['XT'] = my_analysis.loc[0.,"Xppm"]/1000000.
melt_wf["Fe3FeT"] = my_analysis.loc[0.,"Fe3FeT"]

In [13]:
my_models = [['fO2','Kress91']]
my_models = vf.make_df_and_add_model_defaults(my_models)
math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-9.2695394895766

In [14]:
liq_comps=vf.melt_pysulfsat(melt_wf)
result = tb.convert_fe_partition_to_fo2(liq_comps=liq_comps, T_K=pd.Series(PT['T'])+273.15, P_kbar=PT['P']/1000., model="Kress1991", Fe3Fet_Liq=melt_wf["Fe3FeT"],renorm=False)
math.log10(result['fo2_calc'])

overwriting Fe3Fet_Liq to that specified in the function input
(1,)


C:\Users\ehughes\AppData\Local\Temp\ipykernel_25252\3582113947.py:3: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  math.log10(result['fo2_calc'])


-9.274068975767225

And the value for 'Kress91A' is similar, as expected.

In [16]:
my_models = [['fO2','Kress91A']]
my_models = vf.make_df_and_add_model_defaults(my_models)
math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-9.206761031816585

### 'ONeill18' Eq. (9a) in O'Neill et al. (2018) EPSL 504:152-162

Appendix A. Supplementary material - Supplementary data 2. Tab = Table S6 S redox calculator (sulfate,KSOg2=ONeill22.xlsx).

Cell S12: -8.29 [logfO2]

Matches to 2 decimal places. Note spreadsheet uses +273 to convert to K, rather than 273.15 used in VolFe so T in spreadsheet = 1200.15 'C

In [15]:
my_models = [['fO2','ONeill18']]
my_models = vf.make_df_and_add_model_defaults(my_models)

math.log10(vf.f_O2(PT,melt_wf,models=my_models))

-8.289923385626128